In [8]:
housing_features

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
19328,-122.87,38.48,27,3894,776.0,1832,715,3.5085,<1H OCEAN
4806,-118.34,34.02,49,1609,371.0,896,389,2.5156,<1H OCEAN
15645,-122.42,37.79,52,3364,1100.0,2112,1045,2.1343,NEAR BAY
2975,-119.08,35.32,8,11609,2141.0,5696,2100,5.0012,INLAND
18823,-121.62,41.78,40,3272,663.0,1467,553,1.7885,INLAND
...,...,...,...,...,...,...,...,...,...
12239,-116.93,33.73,13,3603,573.0,1644,515,4.0433,INLAND
12181,-117.27,33.77,16,2876,576.0,1859,545,2.0878,<1H OCEAN
18041,-121.98,37.22,46,10088,1910.0,3728,1781,5.2321,<1H OCEAN
7915,-118.08,33.88,26,1507,270.0,931,275,5.1645,<1H OCEAN


In [11]:
import os 
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import  LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score


MODEL_FILE = "model.pkl"
PIPPELINE_FILE = "pipeline.pkl"

#Load Dataset
housing = pd.read_csv("housing.csv")


# create stratified test set3
housing['income_cat'] = pd.cut(housing['median_income'], 
                               bins=[0.0,1.5,3.0,4.5,6.0, np.inf], 
                               labels=[1,2,3,4,5])

def build_pipeline(num_attribs, cat_attribs):
    # for Numercal
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # For categoricals 
    cat_pipeline = Pipeline([
        ("onehor", OneHotEncoder(handle_unknown="ignore")),
    ])

    full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs),
    ])

    return full_pipeline

if not os.path.exists(MODEL_FILE):
    # Load the datasets
    housing = pd.read_csv('housing.csv')
    
    housing['income_cat'] = pd.cut(housing['median_income'], bins=[0.0,1.5,3.0,4.5,6.0, np.inf], labels=[1,2,3,4,5])
    
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
    
    for train_index, test_index in split.split(housing, housing['income_cat']):
        housing = housing.iloc[train_index].drop('income_cat', axis=1)

        # 3. Separate predictors and labels
    housing_labels = housing["median_house_value"].copy()
    # housing_features = housing_features.drop("median_house_value", axis=1)

    num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()
    cat_attribs = ["ocean_proximity"]

    pipeline = build_pipeline(num_attribs, cat_attribs)
    housing_prepared = pipeline.fit_transform(housing_features)
    print(housing_prepared)

        

ValueError: Some column names are not columns of the dataframe: {'median_house_value'}

In [9]:
housing_labels

19328    187800
4806     136600
15645    400000
2975     106300
18823     43500
          ...  
12239    132300
12181    101300
18041    500001
7915     244900
8123     117600
Name: median_house_value, Length: 16512, dtype: int64